# Étude des Patterns et du Fonctionnement des Attaques Adversariales Basées sur le Gradient

## Introduction

Ce notebook explore les attaques adversariales basées sur le gradient, telles que FGSM (Fast Gradient Sign Method) et PGD (Projected Gradient Descent), sur un modèle de classification d'IRM cérébrales. Nous allons visualiser les perturbations introduites par ces attaques et analyser leurs effets.

## Importation des bibliothèques

In [ ]:
!pip install torch torchvision pandas art

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import pandas as pd
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent


## Préparation du Dataset si besoin

In [ ]:
def setup_environment():
    """Setup the environment by checking dataset and renaming images if necessary."""
    if not os.path.exists('adni_dataset2/AugmentedAlzheimerDataset/AD/AD-0001.jpg'):
        if not os.path.exists('adni_dataset2'):
            print('Dataset Missing, run setup.py ? [y/N]')
            choice = input()
            if choice.lower() == 'y':
                os.system('python setup.py')
                print('Fin de setup.py\n')
                print(f"\n Executing {__file__}")
                print("\n ")
            else:
                print("Dataset not found. Exiting.")
                exit()
        print("[*] Renaming images...")
        rename_images_in_directory('adni_dataset2/AugmentedAlzheimerDataset/AD', 'AD')
        rename_images_in_directory('adni_dataset2/AugmentedAlzheimerDataset/CN', 'CN')
        rename_images_in_directory('adni_dataset2/AugmentedAlzheimerDataset/EMCI', 'EMCI')
        rename_images_in_directory('adni_dataset2/AugmentedAlzheimerDataset/LMCI', 'LMCI')

def rename_images_in_directory(directory_path, prefix):
    """Rename images in the directory with a given prefix."""
    files = sorted(os.listdir(directory_path))
    for counter, filename in enumerate(files, start=1):
        new_name = f"{prefix}-{counter:04d}{os.path.splitext(filename)[1]}"
        os.rename(os.path.join(directory_path, filename), os.path.join(directory_path, new_name))

def create_csv_if_not_exists():
    if not os.path.exists('adni_dataset2/train.csv'):
        process_images_to_csv('adni_dataset2/AugmentedAlzheimerDataset/AD', 'adni_dataset2/train.csv')
        process_images_to_csv('adni_dataset2/AugmentedAlzheimerDataset/CN', 'adni_dataset2/train.csv')
        process_images_to_csv('adni_dataset2/AugmentedAlzheimerDataset/EMCI', 'adni_dataset2/train.csv')
        process_images_to_csv('adni_dataset2/AugmentedAlzheimerDataset/LMCI', 'adni_dataset2/train.csv')
        print(f"Les données ont été enregistrées dans adni_dataset2/train.csv")
        print("\n[ ] ------------------------------------------------------------")

def process_images_to_csv(directory_path, output_csv_path):
    """Process images in the directory and write to a CSV file."""
    with open(output_csv_path, mode='a', newline='') as csv_file:
        fieldnames = ['id_code', 'diagnosis']
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        if csv_file.tell() == 0:
            writer.writeheader()
        for filename in sorted(os.listdir(directory_path)):
            name_without_extension = os.path.splitext(filename)[0]
            diagnosis = {'AD': 3, 'LMCI': 2, 'EMCI': 1, 'CN': 0}.get(name_without_extension.split('-')[0], -1)
            writer.writerow({'id_code': name_without_extension, 'diagnosis': diagnosis})

def shuffle_csv(input_csv):
    """Shuffle the CSV file."""
    df = pd.read_csv(input_csv)
    df = df.sample(frac=1).reset_index(drop=True)
    df.to_csv(input_csv, index=False)

## Configuration des paramètres

In [ ]:
EPSILONS = [0.01, 0.02, 0.05, 0.1, 0.2, 0.4, 0.7, 0.9]  # Différentes valeurs d'epsilon à tester
MODEL_PATH = 'model/brain_mri_model.pth'


## Définition du Dataset


In [ ]:
class BrainMRIDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        img_name = self.annotations.iloc[index, 0]
        folder_name = img_name.split('-')[0]
        img_path = os.path.join(self.root_dir, folder_name, img_name + '.jpg')
        image = Image.open(img_path).convert('L')
        y_label = torch.tensor(self.annotations.iloc[index, 1])
        if self.transform:
            image = self.transform(image)
        return image, y_label


## Préparation des données


In [ ]:
def prepare_data(csv_file, root_dir, transform):
    dataset = BrainMRIDataset(csv_file=csv_file, root_dir=root_dir, transform=transform)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    return train_loader, test_loader, train_dataset, test_dataset


## Modèle


In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        dummy_input = torch.zeros(1, 1, 200, 190)
        dummy_output = self.pool(torch.relu(self.conv2(self.pool(torch.relu(self.conv1(dummy_input))))))
        flattened_size = dummy_output.view(-1).shape[0]
        self.fc1 = nn.Linear(flattened_size, 512)
        self.fc2 = nn.Linear(512, 4)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        return self.fc2(x)


## Paramétrage cuda


In [ ]:
def set_device():
    """Set the device for training."""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Chargement du modèle

In [ ]:
def load_model(model, path=MODEL_PATH, device='cpu'):
    """Load the model from a file."""
    model.load_state_dict(torch.load(path, map_location=device))
    print(f"Model loaded from {path}")
    return model


## Visualisation des attaques

In [ ]:
def visualize_attacks(model, test_loader, device, art_classifier, epsilons, num_examples=5):
    model.eval()
    os.makedirs("img/attacks", exist_ok=True)
    inputs, labels = next(iter(test_loader))
    inputs, labels = inputs.to(device), labels.to(device)

    attack_names = ['FGSM', 'PGD']
    attack_results = {}
    perturbation_norms = {'FGSM': [], 'PGD': []}

    for eps in epsilons:
        attack_results[eps] = {}
        # Initialize attacks
        fgsm = FastGradientMethod(art_classifier, eps=eps)
        pgd = ProjectedGradientDescent(art_classifier, eps=eps, max_iter=10)

        # Generate adversarial examples
        x_adv_fgsm = torch.FloatTensor(fgsm.generate(inputs.cpu().numpy())).to(device)
        x_adv_pgd = torch.FloatTensor(pgd.generate(inputs.cpu().numpy())).to(device)

        attack_results[eps]['FGSM'] = x_adv_fgsm
        attack_results[eps]['PGD'] = x_adv_pgd

        # Calculate perturbation norms
        perturbation_norms['FGSM'].append(np.linalg.norm((x_adv_fgsm - inputs).cpu().numpy().reshape(inputs.size(0), -1), axis=1).mean())
        perturbation_norms['PGD'].append(np.linalg.norm((x_adv_pgd - inputs).cpu().numpy().reshape(inputs.size(0), -1), axis=1).mean())

    indices = np.random.choice(len(inputs), num_examples, replace=False)

    for i, idx in enumerate(indices):
        fig, axs = plt.subplots(len(epsilons), 7, figsize=(28, 4 * len(epsilons)))
        original_img = inputs[idx].cpu().squeeze().numpy()

        for j, eps in enumerate(epsilons):
            fgsm_img = attack_results[eps]['FGSM'][idx].cpu().squeeze().numpy()
            pgd_img = attack_results[eps]['PGD'][idx].cpu().squeeze().numpy()

            fgsm_diff = np.abs(original_img - fgsm_img)
            pgd_diff = np.abs(original_img - pgd_img)

            if j == 0:
                axs[j, 0].imshow(original_img, cmap='gray')
                axs[j, 0].set_title(f"Original (Label: {labels[idx].item()})")
                axs[j, 0].axis('off')

            axs[j, 1].imshow(fgsm_img, cmap='gray')
            axs[j, 1].set_title(f"FGSM Perturbed (ε={eps})")
            axs[j, 1].axis('off')

            axs[j, 2].imshow(fgsm_diff, cmap='hot')
            axs[j, 2].set_title(f"FGSM Difference (ε={eps})")
            axs[j, 2].axis('off')

            axs[j, 3].hist(fgsm_diff.flatten(), bins=50, color='red', alpha=0.7)
            axs[j, 3].set_title(f"FGSM Diff Histogram (ε={eps})")

            axs[j, 4].imshow(pgd_img, cmap='gray')
            axs[j, 4].set_title(f"PGD Perturbed (ε={eps})")
            axs[j, 4].axis('off')

            axs[j, 5].imshow(pgd_diff, cmap='hot')
            axs[j, 5].set_title(f"PGD Difference (ε={eps})")
            axs[j, 5].axis('off')

            axs[j, 6].hist(pgd_diff.flatten(), bins=50, color='blue', alpha=0.7)
            axs[j, 6].set_title(f"PGD Diff Histogram (ε={eps})")

        plt.tight_layout()
        plt.savefig(f'img/attacks/attack_example_{i}.png')
        plt.show()

    # Plot perturbation norms
    plt.figure(figsize=(10, 5))
    for attack_name in perturbation_norms:
        plt.plot(epsilons, perturbation_norms[attack_name], label=attack_name)
    plt.xlabel('Epsilon')
    plt.ylabel('Average Perturbation Norm')
    plt.title('Perturbation Norms for Different Epsilon Values')
    plt.legend()
    plt.savefig('img/attacks/perturbation_norms.png')
    plt.show()


## Fonction principale


In [ ]:
def main():
    device = set_device()

    transform = transforms.Compose([
        transforms.Resize((200, 190)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # Prepare data
    train_loader, test_loader, train_dataset, test_dataset = prepare_data(
        'adni_dataset2/train.csv', 'adni_dataset2/AugmentedAlzheimerDataset', transform)

    # Initialize model
    model = Net().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.004, momentum=0.9)

    # Load pre-trained model
    model = load_model(model, device=device)

    # Create ART classifier
    art_classifier = PyTorchClassifier(
        model=model,
        loss=criterion,
        optimizer=optimizer,
        input_shape=(1, 200, 190),
        nb_classes=4,
        clip_values=(0, 1)
    )

    # Visualize attacks
    print("\n[*] Generating attack visualizations...")
    visualize_attacks(model, test_loader, device, art_classifier, EPSILONS)

if __name__ == "__main__":
    main()
